<a href="https://colab.research.google.com/github/mirdbg/Entrega_RAG/blob/main/notebook_entrega/Entrega_RAG_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agente investigador sobre informes 10-K
**LLMs aplicados a Finanzas · MIAX · Entrega final**

Este notebook documenta y reproduce el sistema entregado. La idea central de la práctica es que **recuperar texto no es la arquitectura completa**: el agente debe decidir entre una fuente numérica exacta (XBRL), retrieval textual, lectura completa de sección y comprobación de cobertura. Además, se evalúa no solo la respuesta, sino también **el camino seguido, el coste y la latencia**.

La implementación principal vive en `agente_10k.py`. Los ficheros `miax_s1.py` y `miax_s2.py` se conservan como auxiliares de las sesiones y referencia del baseline. El notebook se centra en reproducibilidad, diagnóstico, ablación y comparación experimental.

## 1. Entorno reproducible

La celda siguiente instala las dependencias utilizadas. Las credenciales no se guardan en el repositorio: en Colab se leen desde **Secrets**. El código busca los ficheros del proyecto en el directorio actual y, como comodidad para Colab, también en la carpeta de Drive usada durante el desarrollo.

In [1]:
%pip install -q \
  langchain==1.3.18 langchain-core==1.6.1 langgraph==1.2.11 \
  langchain-huggingface==1.2.2 langchain-google-genai \
  sentence-transformers==6.0.1 faiss-cpu==1.15.0 rank-bm25==0.2.2 \
  pandas pyarrow

In [2]:
from pathlib import Path
import os, sys, json, time, hashlib, zipfile, importlib
import numpy as np
import pandas as pd

try:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    try:
        os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    except Exception:
        pass
except ImportError:
    pass

# 1) clon limpio / ejecución local; 2) carpeta usada durante el desarrollo en Colab.
CANDIDATOS_PROYECTO = [
    Path.cwd(),
    Path('/content/drive/MyDrive/MIAX_Taller_NLP'),
]
PROJECT_DIR = next((p for p in CANDIDATOS_PROYECTO if (p/'miax_s2.py').is_file()), None)
assert PROJECT_DIR is not None, 'No encuentro miax_s2.py. Ejecuta el notebook desde la raíz del repositorio.'
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

assert os.environ.get('GOOGLE_API_KEY'), 'Configura GOOGLE_API_KEY en Colab Secrets o como variable de entorno.'
print('Proyecto:', PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Proyecto: /content/drive/MyDrive/MIAX_Taller_NLP


## 2. Corpus e integridad

El corpus entregado contiene 48 secciones de seis compañías y dos ejercicios, 1.749 chunks, hechos XBRL y un índice FAISS. La alineación entre índice y metadatos es crítica: un vector asociado a una fila incorrecta produciría un error silencioso.

Los ZIP no se versionan con el código. Se buscan en `data/`, en la raíz del proyecto o en la carpeta de Drive. Se verifican sus hashes antes de extraerlos en `/content/corpus`.

In [3]:
CORPUS_DIR = Path('/content/corpus') if Path('/content').exists() else PROJECT_DIR/'corpus'
PAQUETES = {
    'corpus_miax_2026.zip': '4233c37fc9e9d12091af7a146063ad70903a3fe51404a485854f4021c63daee4',
    'indice_faiss.zip': '6b5610ad8ac6ea50364445d39bb464d993cbd87048fb07c4fe16657d7ac11655',
}

def sha256(ruta):
    h=hashlib.sha256()
    with open(ruta,'rb') as f:
        for bloque in iter(lambda:f.read(1<<20), b''): h.update(bloque)
    return h.hexdigest()

def localizar(nombre):
    candidatos=[PROJECT_DIR/'data'/nombre, PROJECT_DIR/nombre,
                Path('/content/drive/MyDrive/MIAX_Taller_NLP')/nombre]
    return next((p for p in candidatos if p.is_file()), None)

CORPUS_DIR.mkdir(parents=True, exist_ok=True)
for nombre, esperado in PAQUETES.items():
    ruta=localizar(nombre)
    assert ruta is not None, f'No encuentro {nombre}. Colócalo en data/ o junto al notebook.'
    assert sha256(ruta)==esperado, f'Hash incorrecto: {nombre}'
    with zipfile.ZipFile(ruta) as zf: zf.extractall(CORPUS_DIR)

assert (CORPUS_DIR/'chunks.jsonl').is_file()
assert (CORPUS_DIR/'indice/corpus.faiss').is_file()
print('Corpus preparado en', CORPUS_DIR)

Corpus preparado en /content/corpus


In [4]:
import miax_s1, miax_s2
importlib.reload(miax_s1); importlib.reload(miax_s2)
indice, meta, _ = miax_s2.cargar_indice()
secciones, chunks = miax_s2.cargar_corpus()
xbrl = pd.read_parquet(CORPUS_DIR/'xbrl_facts.parquet')
assert indice.ntotal == len(meta) == len(chunks)
print(f'{len(secciones)} secciones · {len(chunks)} chunks · {len(xbrl)} hechos XBRL')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

48 secciones · 1749 chunks · 135 hechos XBRL


In [5]:
# ============================================================
# Exploración del corpus
# ============================================================

df_secciones = pd.DataFrame(secciones)
df_chunks = pd.DataFrame(chunks)
df_meta = meta.copy()
df_xbrl = xbrl.copy()

print("Dimensiones")
print("-" * 50)
print(f"Secciones:       {df_secciones.shape}")
print(f"Chunks:          {df_chunks.shape}")
print(f"Metadatos FAISS: {df_meta.shape}")
print(f"Hechos XBRL:     {df_xbrl.shape}")

print("\nColumnas")
print("-" * 50)
print("secciones :", df_secciones.columns.tolist())
print("chunks    :", df_chunks.columns.tolist())
print("meta      :", df_meta.columns.tolist())
print("xbrl      :", df_xbrl.columns.tolist())

Dimensiones
--------------------------------------------------
Secciones:       (48, 11)
Chunks:          (1749, 10)
Metadatos FAISS: (1749, 10)
Hechos XBRL:     (135, 8)

Columnas
--------------------------------------------------
secciones : ['ticker', 'empresa', 'cik', 'fiscal_year', 'item', 'titulo', 'texto', 'n_tokens', 'url_origen', 'accession', 'item_origen']
chunks    : ['chunk_id', 'ticker', 'fiscal_year', 'item', 'posicion', 'texto', 'n_tokens', 'contiene_tabla', 'inicio_car', 'fin_car']
meta      : ['chunk_id', 'ticker', 'fiscal_year', 'item', 'posicion', 'texto', 'n_tokens', 'contiene_tabla', 'inicio_car', 'fin_car']
xbrl      : ['ticker', 'cik', 'fiscal_year', 'concept', 'value', 'unit', 'period_end', 'form']


In [6]:
display(df_secciones.head(3))
display(df_chunks.head(5))
display(df_xbrl.head(10))

,ticker,empresa,cik,fiscal_year,item,titulo,texto,n_tokens,url_origen,accession,item_origen
0,NVDA,NVIDIA CORP,0001045810,2024,1A,Risk Factors,Item 1A. Risk Factors\n\nThe following risk fa...,18681,https://www.sec.gov/Archives/edgar/data/104581...,0001045810-24-000029,1A
1,NVDA,NVIDIA CORP,0001045810,2024,7,Management's Discussion and Analysis of Financ...,Item 7. Management's Discussion and Analysis o...,8348,https://www.sec.gov/Archives/edgar/data/104581...,0001045810-24-000029,7
2,NVDA,NVIDIA CORP,0001045810,2024,7A,Quantitative and Qualitative Disclosures About...,Item 7A. Quantitative and Qualitative Disclosu...,635,https://www.sec.gov/Archives/edgar/data/104581...,0001045810-24-000029,7A


,chunk_id,ticker,fiscal_year,item,posicion,texto,n_tokens,contiene_tabla,inicio_car,fin_car
0,NVDA-2024-1A-0000,NVDA,2024,1A,0,Item 1A. Risk Factors\n\nThe following risk fa...,475,False,0,2645
1,NVDA-2024-1A-0001,NVDA,2024,1A,1,"Risks Related to Regulatory, Legal, Our Stock ...",278,False,2647,4212
2,NVDA-2024-1A-0002,NVDA,2024,1A,2,Risks Related to Our Industry and Markets\n\nF...,500,False,4214,7171
3,NVDA-2024-1A-0003,NVDA,2024,1A,3,may be harmed.\n\nWe have begun offering enter...,500,False,6705,9596
4,NVDA-2024-1A-0004,NVDA,2024,1A,4,our customers have in-house expertise and inte...,99,False,9133,9708


,ticker,cik,fiscal_year,concept,value,unit,period_end,form
0,AAPL,0000320193,2024,Assets,3.649800e+11,USD,2024-09-28,10-K
1,AAPL,0000320193,2024,CashAndCashEquivalentsAtCarryingValue,2.994300e+10,USD,2024-09-28,10-K
2,AAPL,0000320193,2024,EarningsPerShareBasic,6.110000e+00,USD/shares,2024-09-28,10-K
3,AAPL,0000320193,2024,EarningsPerShareDiluted,6.080000e+00,USD/shares,2024-09-28,10-K
4,AAPL,0000320193,2024,GrossProfit,1.806830e+11,USD,2024-09-28,10-K
5,AAPL,0000320193,2024,Liabilities,3.080300e+11,USD,2024-09-28,10-K
6,AAPL,0000320193,2024,NetCashProvidedByUsedInOperatingActivities,1.182540e+11,USD,2024-09-28,10-K
7,AAPL,0000320193,2024,NetIncomeLoss,9.373600e+10,USD,2024-09-28,10-K
8,AAPL,0000320193,2024,OperatingIncomeLoss,1.232160e+11,USD,2024-09-28,10-K
9,AAPL,0000320193,2024,ResearchAndDevelopmentExpense,3.137000e+10,USD,2024-09-28,10-K


In [7]:
cobertura = (
    df_secciones
    .groupby(["ticker", "fiscal_year"])["item"]
    .apply(lambda x: ", ".join(sorted(x.astype(str).unique())))
    .reset_index(name="items_disponibles")
)

display(cobertura)

,ticker,fiscal_year,items_disponibles
0,AAPL,2024,"1A, 7, 7A, 8"
1,AAPL,2025,"1A, 7, 7A, 8"
2,AMZN,2024,"1A, 7, 7A, 8"
3,AMZN,2025,"1A, 7, 7A, 8"
4,GOOGL,2024,"1A, 7, 7A, 8"
5,GOOGL,2025,"1A, 7, 7A, 8"
6,META,2024,"1A, 7, 7A, 8"
7,META,2025,"1A, 7, 7A, 8"
8,MSFT,2024,"1A, 7, 7A, 8"
9,MSFT,2025,"1A, 7, 7A, 8"


In [8]:
# Cuántos chunks hay en cada sección de cada empresa

pd.crosstab(
    df_chunks["ticker"],
    df_chunks["item"],
    margins=True
)

item,1A,7,7A,8,All
ticker,,,,,
AAPL,67,21,4,87,179
AMZN,60,53,10,154,277
GOOGL,77,60,10,166,313
META,167,70,7,165,409
MSFT,65,54,2,152,273
NVDA,97,49,4,148,298
All,533,307,37,872,1749


In [9]:
pd.set_option("display.max_colwidth", None)

display(
    df_chunks[(df_chunks["item"] == "7A") & (df_chunks["ticker"] == "MSFT")]['texto'].head(1)
)

pd.reset_option("display.max_colwidth")


,texto
359,"ITEM 7A. QUANTITATIVE AND QUALITATIVE DISCLOSURES ABOUT MARKET RISK\n\nRISKS\n\nWe are exposed to economic risk from foreign exchange rates, interest rates, credit risk, and equity prices. We use derivatives instruments to manage these risks, however, they may still impact our consolidated financial statements.\n\nForeign Currencies\n\nCertain forecasted transactions, assets, and liabilities are exposed to foreign currency risk. We monitor our foreign currency exposures daily to maximize the economic effectiveness of our foreign currency positions, including hedges. Principal currency exposures include the Euro, Japanese yen, British pound, Canadian dollar, and Australian dollar.\n\nInterest Rate\n\nSecurities held in our fixed-income portfolio are subject to different interest rate risks based on their maturities. We manage the average maturity of the fixed-income portfolio to achieve economic returns that correlate to certain global fixed-income indices.\n\nCredit\n\nOur fixed-income portfolio is diversified and consists primarily of investment-grade securities. We manage credit exposures relative to broad-based indices to facilitate portfolio diversification.\n\nEquity\n\nSecurities held in our equity investments portfolio are subject to price risk.\n\nSENSITIVITY ANALYSIS\n\nThe following table sets forth the potential loss in future earnings or fair values, including associated derivatives, resulting from hypothetical changes in relevant market rates or prices:\n\n(In millions)\nRisk Categories\t\tHypothetical Change\t\tJune 30, 2024\t\t\tImpact\nForeign currency – Revenue\t\t10% decrease in foreign exchange rates\t\t$\t(9,605\t)\t\t\tEarnings\nForeign currency – Investments\t\t10% decrease in foreign exchange rates\t\t\t(38\t)\t\t\tFair Value\nInterest rate\t\t100 basis point increase in U.S. treasury interest rates\t\t\t(1,343\t)\t\t\tFair Value\nCredit\t\t100 basis point increase in credit spreads\t\t\t(318\t)\t\t\tFair Value\nEquity\t\t10% decrease in equity market prices\t\t\t(1,078\t)\t\t\tEarnings"


In [10]:
pd.set_option("display.max_colwidth", None)

display(
    df_chunks[(df_chunks["item"] == "7A") & (df_chunks["ticker"] == "GOOGL")]['texto'].head(1)
)

pd.reset_option("display.max_colwidth")



,texto
819,"ITEM 7A.QUANTITATIVE AND QUALITATIVE DISCLOSURES ABOUT MARKET RISK\n\nWe are exposed to financial market risks, including changes in foreign currency exchange rates, interest rates, and equity investment risks.\n\nForeign Currency Exchange Risk\n\nWe transact business globally in multiple currencies. International revenues, as well as costs and expenses denominated in foreign currencies, expose us to the risk of fluctuations in foreign currency exchange rates against the U.S. dollar. As discussed below, we enter into derivative instruments to hedge foreign currency risk. Principal currencies hedged included the Australian dollar, British pound, Canadian dollar, Euro, and Japanese yen. For the purpose of analyzing foreign currency exchange risk, we considered the historical trends in foreign currency exchange rates and determined that it was reasonably possible that adverse changes in exchange rates of 10% could be experienced.\n\nWe use foreign currency forward and option contracts to offset the foreign exchange risk on monetary assets and liabilities denominated in currencies other than the functional currency of the subsidiary. These forward and option contracts reduce, but do not entirely eliminate, the effect of foreign currency exchange rate movements on our assets and liabilities. The foreign currency gains and losses on these assets and liabilities are recorded in OI&E, which are offset by the gains and losses on the forward and option contracts.\n\nIf an adverse 10% foreign currency exchange rate change was applied to net monetary assets, liabilities, and commitments denominated in currencies other than the functional currencies at the balance sheet date, it would have resulted in an adverse effect on income before income taxes of approximately $503 million and $135 million as of December 31, 2023 and 2024, respectively, after consideration of the effect of foreign exchange contracts in place for the years ended December 31, 2023 and 2024.\n\nWe use foreign currency forward and option contracts, including collars (an option strategy comprised of a combination of purchased and written options) to protect forecasted U.S. dollar-equivalent earnings from changes in foreign currency exchange rates. When the U.S. dollar strengthens, gains from foreign currency forward and option contacts reduce the foreign currency losses related to our earnings. When the U.S. dollar weakens, losses from foreign currency forward and option contracts offset the foreign currency gains related to our earnings. These hedging contracts reduce, but do not entirely eliminate, the effect of foreign currency exchange rate movements. We designate these contracts as cash flow hedges for accounting purposes. We reflect the gains and losses of foreign currency spot rate changes as a component of accumulated other comprehensive income (AOCI) and subsequently reclassify them into revenues to offset the hedged exposures as they occur.\n\nIf the U.S. dollar weakened by"


In [11]:
# Qué conceptos existen por empresa

conceptos_por_empresa = (
    df_xbrl
    .groupby("ticker")["concept"]
    .apply(lambda x: sorted(x.unique()))
)

for ticker, conceptos in conceptos_por_empresa.items():
    print(f"\n{ticker} ({len(conceptos)} conceptos)")
    print(", ".join(conceptos))


AAPL (12 conceptos)
Assets, CashAndCashEquivalentsAtCarryingValue, EarningsPerShareBasic, EarningsPerShareDiluted, GrossProfit, Liabilities, NetCashProvidedByUsedInOperatingActivities, NetIncomeLoss, OperatingIncomeLoss, ResearchAndDevelopmentExpense, RevenueFromContractWithCustomerExcludingAssessedTax, StockholdersEquity

AMZN (9 conceptos)
Assets, CashAndCashEquivalentsAtCarryingValue, EarningsPerShareBasic, EarningsPerShareDiluted, NetCashProvidedByUsedInOperatingActivities, NetIncomeLoss, OperatingIncomeLoss, RevenueFromContractWithCustomerExcludingAssessedTax, StockholdersEquity

GOOGL (12 conceptos)
Assets, CashAndCashEquivalentsAtCarryingValue, EarningsPerShareBasic, EarningsPerShareDiluted, Liabilities, NetCashProvidedByUsedInOperatingActivities, NetIncomeLoss, OperatingIncomeLoss, ResearchAndDevelopmentExpense, RevenueFromContractWithCustomerExcludingAssessedTax, Revenues, StockholdersEquity

META (11 conceptos)
Assets, CashAndCashEquivalentsAtCarryingValue, EarningsPerShareB

In [12]:
# Disponibilidad de los conceptos

disponibilidad_xbrl = pd.crosstab(
    df_xbrl["concept"],
    df_xbrl["ticker"]
)

display(disponibilidad_xbrl)

ticker,AAPL,AMZN,GOOGL,META,MSFT,NVDA
concept,,,,,,
Assets,2,2,2,2,2,2
CashAndCashEquivalentsAtCarryingValue,2,2,2,2,2,2
EarningsPerShareBasic,2,2,2,2,2,2
EarningsPerShareDiluted,2,2,2,2,2,2
GrossProfit,2,0,0,0,2,2
Liabilities,2,0,2,2,2,2
NetCashProvidedByUsedInOperatingActivities,2,2,2,2,2,2
NetIncomeLoss,2,2,2,2,2,2
OperatingIncomeLoss,2,2,2,2,2,2


In [13]:
display(
    df_xbrl[
        df_xbrl["concept"].isin([
            "Revenues",
            "RevenueFromContractWithCustomerExcludingAssessedTax",
            "GrossProfit",
            "NetIncomeLoss"
        ])
    ][
        ["ticker", "fiscal_year", "concept", "value", "unit"]
    ].sort_values(["ticker", "fiscal_year", "concept"])
)

,ticker,fiscal_year,concept,value,unit
4,AAPL,2024,GrossProfit,1.806830e+11,USD
7,AAPL,2024,NetIncomeLoss,9.373600e+10,USD
10,AAPL,2024,RevenueFromContractWithCustomerExcludingAssess...,3.910350e+11,USD
16,AAPL,2025,GrossProfit,1.952010e+11,USD
19,AAPL,2025,NetIncomeLoss,1.120100e+11,USD
22,AAPL,2025,RevenueFromContractWithCustomerExcludingAssess...,4.161610e+11,USD
29,AMZN,2024,NetIncomeLoss,5.924800e+10,USD
31,AMZN,2024,RevenueFromContractWithCustomerExcludingAssess...,6.379590e+11,USD
38,AMZN,2025,NetIncomeLoss,7.767000e+10,USD
40,AMZN,2025,RevenueFromContractWithCustomerExcludingAssess...,7.169240e+11,USD


In [14]:
# Analizamos el FAISS

print("Tipo de índice:", type(indice).__name__)
print("Número de vectores:", indice.ntotal)
print("Dimensión de los embeddings:", indice.d)

print("\nEjemplo de metadatos asociados a un vector:")
display(meta.head(3))

Tipo de índice: IndexFlatIP
Número de vectores: 1749
Dimensión de los embeddings: 384

Ejemplo de metadatos asociados a un vector:


,chunk_id,ticker,fiscal_year,item,posicion,texto,n_tokens,contiene_tabla,inicio_car,fin_car
0,NVDA-2024-1A-0000,NVDA,2024,1A,0,Item 1A. Risk Factors\n\nThe following risk fa...,475,False,0,2645
1,NVDA-2024-1A-0001,NVDA,2024,1A,1,"Risks Related to Regulatory, Legal, Our Stock ...",278,False,2647,4212
2,NVDA-2024-1A-0002,NVDA,2024,1A,2,Risks Related to Our Industry and Markets\n\nF...,500,False,4214,7171


In [15]:
# Una búsqueda manual de prueba a ver qué saldría

query = "risks related to artificial intelligence"

vector_query = miax_s2.codificar([query])

scores, posiciones = indice.search(vector_query, 5)

resultados = meta.iloc[posiciones[0]].copy()
resultados["similitud"] = scores[0]
resultados["texto"] = resultados["texto"].str[:300] + "..."

display(
    resultados[
        ["chunk_id", "ticker", "fiscal_year", "item", "similitud", "texto"]
    ]
)

,chunk_id,ticker,fiscal_year,item,similitud,texto
920,GOOGL-2025-1A-0014,GOOGL,2025,1A,0.826109,Risks Related to our Industry\n\nIssues in the...
454,MSFT-2025-1A-0017,MSFT,2025,1A,0.802003,of operations.\n\n\n\nIssues in the developmen...
315,MSFT-2024-1A-0017,MSFT,2024,1A,0.799385,.\n\nIssues in the development and use of AI m...
763,GOOGL-2024-1A-0013,GOOGL,2024,1A,0.797524,Risks Related to our Industry\n\nPeople access...
764,GOOGL-2024-1A-0014,GOOGL,2024,1A,0.796179,AI will present ethical issues and may have br...


In [16]:
# ¿Qué hay del metadato de contiene tabla?

# Vemos 3 ejemplos de chunks que contienen tablas

chunks_tabla = meta[meta["contiene_tabla"] == True]

print("Número de chunks con tabla:", len(chunks_tabla))

for i, (_, chunk) in enumerate(chunks_tabla.head(3).iterrows(), 1):
    print(f"\n{'='*80}")
    print(f"EJEMPLO {i}")
    print(f"{'='*80}")
    print("Ticker:", chunk["ticker"])
    print("Año:", chunk["fiscal_year"])
    print("Item:", chunk["item"])
    print("Tokens:", chunk["n_tokens"])
    print("Contiene tabla:", chunk["contiene_tabla"])
    print("\nTEXTO DEL CHUNK:\n")
    print(chunk["texto"])

Número de chunks con tabla: 721

EJEMPLO 1
Ticker: NVDA
Año: 2024
Item: 7
Tokens: 475
Contiene tabla: True

TEXTO DEL CHUNK:

Year Ended
Jan 28, 2024		Jan 29, 2023		Change
($ in millions, except per share data)
Revenue	$	60,922			$	26,974			Up 126%
Gross margin	72.7	%		56.9	%		Up 15.8 pts
Operating expenses	$	11,329			$	11,132			Up 2%
Operating income	$	32,972			$	4,224			Up 681%
Net income	$	29,760			$	4,368			Up 581%
Net income per diluted share	$	11.93			$	1.74			Up 586%

We specialize in markets where our computing platforms can provide tremendous acceleration for applications. These platforms incorporate processors, interconnects, software, algorithms, systems, and services to deliver unique value. Our platforms address four large markets where our expertise is critical: Data Center, Gaming, Professional Visualization, and Automotive.

Revenue for fiscal year 2024 was $60.9 billion, up 126% from a year ago.

Data Center revenue for fiscal year 2024 was up 217%. Strong demand was d

## 3. Sistema entregado: tools, retrieval y contrato

Las cuatro firmas exigidas se mantienen sin cambios: `list_available`, `get_xbrl_fact`, `search_filings` y `read_section`. Sus docstrings forman parte del routing porque son información que el modelo utiliza para decidir qué herramienta llamar.

La mejora del retrieval se implementa de forma incremental:

1. **Denso baseline**: búsqueda de la sesión 1.
2. **Filtro por metadatos**: restringe ticker, ejercicio e item.
3. **Híbrido BM25 + denso**: fusiona rankings con Reciprocal Rank Fusion (RRF), evitando sumar scores de escalas incompatibles.
4. **Query rewriting**: el propio modelo reescribe la necesidad de información a una consulta breve en inglés y con vocabulario del 10-K.

La versión final usa las cuatro capas; las anteriores se conservan para poder medir qué aporta cada arreglo.

In [17]:
import agente_10k as sistema
importlib.reload(sistema)
print('Tools:', [t.name for t in sistema.HERRAMIENTAS])
print('Modelo:', sistema.MODELO)

Tools: ['list_available', 'get_xbrl_fact', 'search_filings', 'read_section']
Modelo: google_genai:gemini-3.8-flash


### 3.1 Comprobaciones deterministas de las tools

Antes de involucrar al LLM verificamos dos comportamientos que no deberían depender de generación: una cifra se obtiene de XBRL y una ausencia real se comunica explícitamente. Esto evita confundir “el modelo sabe la respuesta” con “el sistema consultó la fuente correcta”.

In [18]:
# La tool básica de datos disponibles

print(sistema.list_available.invoke({}))

AAPL (Apple Inc.): ejercicios [2024, 2025], items ['1A', '7', '7A', '8']
AMZN (AMAZON COM INC): ejercicios [2024, 2025], items ['1A', '7', '7A', '8']
GOOGL (Alphabet Inc.): ejercicios [2024, 2025], items ['1A', '7', '7A', '8']
META (Meta Platforms, Inc.): ejercicios [2024, 2025], items ['1A', '7', '7A', '8']
MSFT (MICROSOFT CORP): ejercicios [2024, 2025], items ['1A', '7', '7A', '8']
NVDA (NVIDIA CORP): ejercicios [2024, 2025], items ['1A', '7', '7A', '8']


In [19]:
# Comrpobamos la tabla de datos

print(sistema.get_xbrl_fact.invoke({'ticker':'NVDA','fiscal_year':2024,'concept':'Revenues'}))
print()
print(sistema.get_xbrl_fact.invoke({'ticker':'AMZN','fiscal_year':2025,'concept':'GrossProfit'}))

NVDA FY2024 Revenues = 60,922,000,000 USD (cierre 2024-01-28, 10-K)

AMZN no reportó 'GrossProfit' en FY2025. Conceptos disponibles: ['Assets', 'CashAndCashEquivalentsAtCarryingValue', 'EarningsPerShareBasic', 'EarningsPerShareDiluted', 'NetCashProvidedByUsedInOperatingActivities', 'NetIncomeLoss', 'OperatingIncomeLoss', 'RevenueFromContractWithCustomerExcludingAssessedTax', 'StockholdersEquity'].


In [20]:
# Conmprobamos search_fillings
# Comprobamos que devuelve 5 chunks

print(
    sistema.search_filings.invoke({
        "query": "operating income by reportable segments",
        "ticker": "NVDA",
        "fiscal_year": 2024,
        "item": "7"
    })
)

[NVDA-2024-7-0014] NVDA FY2024 Item 7 (similitud 0.033)
Note: this fragment contains tabular content.

Year Ended
Jan 28, 2024		Jan 29, 2023
Revenue	100.0	%		100.0	%
Cost of revenue	27.3			43.1
Gross profit	72.7			56.9
Operating expenses
Research and development	14.2			27.2
Sales, general and administrative	4.4			9.1
Acquisition termination cost	—			5.0
Total operating expenses	18.6			41.3
Operating income	54.1			15.6
Interest income	1.4			1.0
Interest expense	(0.4)			(1.0)
Other, net	0.4			(0.1)
Other income (expense), net	1.4			(0.1)
Income before income tax	55.5			15.5
Income tax expense (benefit)	6.6			(0.7)
Net income	48.9	%		16.2	%

Reportable Segments

Revenue by Reportable Segments

Year Ended
Jan 28, 2024		Jan 29, 2023		$ Change		% Change
($ in millions)
Compute & Networking	$	47,405			$	15,068			$	32,337			215	%
Graphics	13,517			11,906			1,611			14	%
Total	$	60,922			$	26,974			$	33,948			126	%

Operating Income by Reportable Segments

---

[NVDA-2024-7-0013] NVDA FY2024 Ite

In [21]:
# Lectura completa

print(
    sistema.read_section.invoke({
        "ticker": "NVDA",
        "fiscal_year": 2024,
        "item": "7"
    })
)

Item 7. Management's Discussion and Analysis of Financial Condition and Results of Operations

The following discussion and analysis of our financial condition and results of operations should be read in conjunction with “Item 1A. Risk Factors”, our Consolidated Financial Statements and related Notes thereto, as well as other cautionary statements and risks described elsewhere in this Annual Report on Form 10-K, before deciding to purchase, hold or sell shares of our common stock.

Overview

Our Company and Our Businesses

NVIDIA pioneered accelerated computing to help solve the most challenging computational problems. Since our original focus on PC graphics, we have expanded to several other large and important computationally intensive fields. NVIDIA has leveraged its GPU architecture to create platforms for accelerated computing, AI solutions, scientific computing, data science, AV, robotics, metaverse and 3D internet applications.

Our two operating segments are "Compute & Networki

## 4. Golden set propio

La entrega exige 20 preguntas conocidas, con al menos seis comparativas. Las familias utilizadas son:

- **numérica**: la cifra se contrasta con XBRL;
- **extractiva**: la verdad se ancla a una frase literal del 10-K;
- **comparativa**: obliga a combinar más de un ejercicio y, cuando corresponde, fuentes numéricas y textuales.

`golden_set.jsonl` es la única fuente de verdad. No se generan anclas a partir del mismo retriever que después se evalúa, porque eso haría circular la evaluación.

In [22]:
RUTA_GOLDEN = PROJECT_DIR/'golden_set.jsonl'
assert RUTA_GOLDEN.is_file(), 'Falta golden_set.jsonl: debe acompañar a la entrega.'
golden=[json.loads(l) for l in RUTA_GOLDEN.open(encoding='utf-8') if l.strip()]

def validar_golden(preguntas):
    problemas=[]
    if len(preguntas)!=20: problemas.append(f'Deben ser 20 preguntas; hay {len(preguntas)}')
    if sum(p.get('familia')=='comparativa' for p in preguntas)<6: problemas.append('Debe haber al menos 6 comparativas')
    ids=[p.get('id') for p in preguntas]
    if len(ids)!=len(set(ids)): problemas.append('Hay IDs repetidos')
    for p in preguntas:
        if p.get('familia') in {'extractiva','comparativa'} and p.get('item_esperado') and not p.get('ancla_texto'):
            problemas.append(f"{p.get('id')}: falta ancla_texto")
    return problemas

pd.set_option("display.max_colwidth", None)


problemas=validar_golden(golden)
assert not problemas, '\n'.join(problemas)
df_golden=pd.DataFrame(golden)
display(df_golden[['id','familia','ticker','fiscal_year','pregunta']])
print('Distribución:', df_golden.familia.value_counts().to_dict())


pd.reset_option("display.max_colwidth")

,id,familia,ticker,fiscal_year,pregunta
0,g3-001,numerica,NVDA,2024,¿Cuál fue el revenue de NVIDIA durante 2024?
1,g3-002,numerica,AAPL,2025,¿Cuánto gastó Apple en investigación y desarrollo en 2025?
2,g3-003,numerica,MSFT,2025,¿Cuál fue el operating income de Microsoft en FY2025?
3,g3-004,numerica,AMZN,2025,¿Cuál fue el gross profit de Amazon en 2025?
4,g3-005,numerica,META,2025,¿Cuál fue la provisión por impuestos sobre beneficios de Meta en 2025?
5,g3-006,numerica,GOOGL,2025,¿Cuál era el total de activos de Alphabet al cierre de FY2025?
6,g3-007,numerica,GOOGL,2025,"Entre NVIDIA, Microsoft y Alphabet, ¿qué compañía obtuvo el mayor beneficio neto en FY2025 y cuál fue ese beneficio?"
7,g3-008,extractiva,NVDA,2025,¿Por qué considera NVIDIA especialmente difícil competir en China en FY2025 en relación con las restricciones regulatorias estadounidenses?
8,g3-009,extractiva,MSFT,2025,¿Qué nuevo riesgo de ciberseguridad identifica Microsoft al incorporar IA generativa a sus propios sistemas internos en 2025?
9,g3-010,extractiva,MSFT,2025,¿Qué efecto tuvo el escalado de la infraestructura de IA sobre el margen de Microsoft Cloud en FY2025 y qué factor lo compensó parcialmente?


Distribución: {'numerica': 7, 'extractiva': 7, 'comparativa': 6}


## 5. Ablación del retrieval

Medimos `recall@5` contra el ancla textual después de cada arreglo. También registramos la posición del primer chunk que contiene el ancla. Así distinguimos un fallo por poco (posición 6) de un fallo estructural (posición muy baja o ausencia).

La reescritura implica una llamada al modelo, por lo que su coste/latencia no debe ocultarse. La tabla separa la calidad del retrieval de su tiempo de ejecución.

In [23]:
def comparar_retrieval(preguntas):
    evaluables=[p for p in preguntas if p.get('ancla_texto')]
    filas=[]
    for p in evaluables:
        q0=p['pregunta']; q1=sistema.reescribir_consulta(q0)
        configs={
            '1_denso': lambda: sistema.buscar_denso_baseline(q0, k=2000),
            '2_filtros': lambda: sistema.buscar_denso_filtrado(q0,p.get('ticker'),p.get('fiscal_year'),p.get('item_esperado'),2000),
            '3_hibrido': lambda: sistema.buscar_hibrido(q0,p.get('ticker'),p.get('fiscal_year'),p.get('item_esperado'),2000),
            '4_rewrite+denso': lambda: sistema.buscar_denso_filtrado(q1,p.get('ticker'),p.get('fiscal_year'),p.get('item_esperado'),2000),
            '5_rewrite+hibrido': lambda: sistema.buscar_hibrido(q1,p.get('ticker'),p.get('fiscal_year'),p.get('item_esperado'),2000),
        }
        fila={'id':p['id']}
        for nombre, fn in configs.items():
            t0=time.perf_counter(); rec=fn(); dt=time.perf_counter()-t0
            pos=miax_s2.posicion_del_ancla(p,rec)
            fila[nombre+'_hit@5']=bool(pos and pos<=5)
            fila[nombre+'_rank']=pos
            fila[nombre+'_ms']=1000*dt
        filas.append(fila)
    return pd.DataFrame(filas)

tabla_ablation=comparar_retrieval(golden)
display(tabla_ablation)
resumen_ablation=[]
for nombre in ['1_denso','2_filtros','3_hibrido','4_rewrite+denso','5_rewrite+hibrido']:
    resumen_ablation.append({
        'configuracion':nombre,
        'recall@5':tabla_ablation[nombre+'_hit@5'].mean(),
        'latencia_retrieval_ms':tabla_ablation[nombre+'_ms'].mean(),
    })
resumen_ablation=pd.DataFrame(resumen_ablation)
display(resumen_ablation.style.format({'recall@5':'{:.1%}','latencia_retrieval_ms':'{:.1f}'}))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,id,1_denso_hit@5,1_denso_rank,1_denso_ms,2_filtros_hit@5,2_filtros_rank,2_filtros_ms,3_hibrido_hit@5,3_hibrido_rank,3_hibrido_ms,4_rewrite+denso_hit@5,4_rewrite+denso_rank,4_rewrite+denso_ms,5_rewrite+hibrido_hit@5,5_rewrite+hibrido_rank,5_rewrite+hibrido_ms
0,g3-008,False,178,1681.157683,False,21,84.968778,False,15,173.416742,False,15,54.444175,False,12,139.252726
1,g3-009,False,76,176.260709,True,4,87.770431,False,13,166.663273,True,3,55.585195,True,1,142.401256
2,g3-010,False,39,207.124413,False,8,79.786725,False,6,174.229166,True,5,46.535886,True,4,133.042205
3,g3-011,False,9,185.105136,True,3,74.805883,True,2,192.342624,True,2,49.486077,True,2,140.599257
4,g3-012,False,1127,190.508643,False,26,82.861781,False,32,177.246278,True,1,47.278177,True,1,138.974653
5,g3-013,False,72,314.640938,True,1,121.501985,True,1,266.579057,True,1,69.772060,True,1,222.210746
6,g3-014,False,154,183.590329,False,6,71.117813,False,7,230.295529,True,5,176.720402,True,3,347.858675
7,g3-015,False,88,202.455073,True,3,99.647873,True,2,201.931727,True,1,48.586806,True,1,142.689634
8,g3-016,False,227,195.293432,False,9,82.770808,False,7,187.516522,True,5,49.454038,True,4,139.829343
9,g3-017,False,27,281.758191,True,3,114.303214,True,2,253.947446,True,3,70.558418,True,4,202.299164


,configuracion,recall@5,latencia_retrieval_ms
0,1_denso,0.0%,327.3
1,2_filtros,46.2%,88.7
2,3_hibrido,46.2%,200.2
3,4_rewrite+denso,84.6%,62.3
4,5_rewrite+hibrido,92.3%,167.6


### 5.1 Un caso visible antes y después

La métrica agregada debe poder explicarse con ejemplos concretos. La función siguiente imprime los top-5 del mismo caso con baseline, filtros e híbrido reescrito. Es útil para detectar si el error era de documento, de vocabulario o de ranking.

In [ ]:
def demo_retrieval(p, k=5):
    q1=sistema.reescribir_consulta(p['pregunta'])
    variantes={
        'DENSO': sistema.buscar_denso_baseline(p['pregunta'], k=k),
        '+ FILTROS': sistema.buscar_denso_filtrado(p['pregunta'],p.get('ticker'),p.get('fiscal_year'),p.get('item_esperado'),k),
        '+ REWRITE + HIBRIDO': sistema.buscar_hibrido(q1,p.get('ticker'),p.get('fiscal_year'),p.get('item_esperado'),k),
    }
    print('Pregunta:',p['pregunta']); print('Rewrite:',q1)
    for nombre, rec in variantes.items():
        print('\n',nombre)
        for j,f in enumerate(rec,1):
            hit=miax_s2.acierta(p,[f])
            print(f"{j}. {'✓' if hit else ' '} {f['ticker']} FY{f['fiscal_year']} Item {f['item']} {f['chunk_id']}")

caso=next(p for p in golden if p.get('ancla_texto'))
demo_retrieval(caso)

Pregunta: ¿Por qué considera NVIDIA especialmente difícil competir en China en FY2025 en relación con las restricciones regulatorias estadounidenses?
Rewrite: NVIDIA FY2025 China US export controls restrictions compete

 DENSO
1.   NVDA FY2025 Item 1A NVDA-2025-1A-0034
2.   NVDA FY2024 Item 7 NVDA-2024-7-0000
3.   NVDA FY2024 Item 1A NVDA-2024-1A-0035
4.   NVDA FY2025 Item 7 NVDA-2025-7-0000
5.   NVDA FY2025 Item 1A NVDA-2025-1A-0036

 + FILTROS
1.   NVDA FY2025 Item 1A NVDA-2025-1A-0034
2.   NVDA FY2025 Item 1A NVDA-2025-1A-0036
3.   NVDA FY2025 Item 1A NVDA-2025-1A-0037
4.   NVDA FY2025 Item 1A NVDA-2025-1A-0031
5.   NVDA FY2025 Item 1A NVDA-2025-1A-0003

 + REWRITE + HIBRIDO
1.   NVDA FY2025 Item 1A NVDA-2025-1A-0037
2.   NVDA FY2025 Item 1A NVDA-2025-1A-0031
3.   NVDA FY2025 Item 1A NVDA-2025-1A-0038
4.   NVDA FY2025 Item 1A NVDA-2025-1A-0032
5.   NVDA FY2025 Item 1A NVDA-2025-1A-0036


## 6. Agente, guardrails y salida estructurada

El agente final mantiene el contrato `RespuestaFinanciera` del enunciado. El routing esperado es asimétrico: XBRL para cifras; retrieval para contenido cualitativo; `read_section` solo como fallback caro; y `list_available` para comprobar cobertura.

Se añaden tres protecciones:

- límite de llamadas a herramientas;
- límite de llamadas al modelo;
- middleware propio de verificación XBRL. El middleware inspecciona la respuesta estructurada, ejecuta el extractor de cifras y, si la cifra principal no coincide con ningún hecho XBRL del ticker/ejercicio dentro de una tolerancia del 1 %, devuelve el desajuste al modelo para una única corrección.

In [ ]:
agente=sistema.construir_agente_final()
assert [t.name for t in sistema.HERRAMIENTAS]==['list_available','get_xbrl_fact','search_filings','read_section']
print('Agente final construido y contrato de tools verificado.')

Agente final construido y contrato de tools verificado.


In [ ]:
with open(PROJECT_DIR / "agente_10k.py", encoding="utf-8") as f:
    contenido = f.read()

# Busca y muestra solo la función que nos interesa
inicio = contenido.find("def verificar_cifras_contra_xbrl")
print(contenido[inicio-20:inicio+2000])

_jump_to=["model"])
def verificar_cifras_contra_xbrl(state: AgentState, runtime: Runtime) -> dict | None:
    """Rechaza una cifra estructurada que no coincida con ningún hecho XBRL.

    Solo aplica cuando la respuesta declara XBRL como fuente (``fuente == 'xbrl'``).
    Si la cifra viene del texto (``fuente == 'texto'``), no tiene sentido exigir
    que coincida con un hecho XBRL que, por construcción, puede no existir para
    ese concepto/empresa — sería penalizar el comportamiento correcto de usar el
    texto cuando XBRL no reporta el dato.

    Además ejecuta el extractor de cifras sobre la prosa para dejar explícita la
    inspección requerida por la práctica. La decisión de corrección se basa en
    ``cifra`` porque es el campo contractual inequívoco de la magnitud principal.
    """
    respuesta = state.get("structured_response")
    if respuesta is None:
        return None
    _ = miax_s2.extraer_cifras(getattr(respuesta, "respuesta", "") or "")
    cifra = getattr(respues

### 6.1 Smoke tests y trayectoria

Probamos una pregunta numérica y una cualitativa. La respuesta final no es suficiente: imprimimos la trayectoria para comprobar que la cifra pasa por XBRL y que la pregunta cualitativa usa retrieval.

In [ ]:
for i,pregunta in enumerate([
    '¿Cuál fue el revenue de NVIDIA en FY2024?',
    '¿Qué riesgos relacionados con IA menciona Microsoft en FY2025?',
],1):
    print('\n'+'='*90+'\n',pregunta)
    r=sistema.responder(pregunta, thread_id=f'smoke-{i}')
    miax_s2.pretty_trace(r)


 ¿Cuál fue el revenue de NVIDIA en FY2024?
  1. get_xbrl_fact(ticker='NVDA', concept='RevenueFromContractWithCustomerExcludingAssessedTax', fiscal_year=2024)
       -> NVDA no reportó 'RevenueFromContractWithCustomerExcludingAssessedTax' en FY2024. Conceptos disponibles: ['Assets', 'CashAndCashEquivalentsAtCarryingValue', 'EarningsPerShareBasic', 'EarningsPerShareDiluted', 'GrossProfit…
  2. get_xbrl_fact(ticker='NVDA', concept='Revenues', fiscal_year=2024)
       -> NVDA FY2024 Revenues = 60,922,000,000 USD (cierre 2024-01-28, 10-K)
  3. RespuestaFinanciera(fuente='xbrl', unidad='USD', ejercicio=2024, respuesta='El revenue (ingresos totales) de NVIDIA en el ejercicio fiscal 2024 (FY2024) fue de 60.922.000.000 USD (60.922 millones de dólares).', ticker='NVDA', cifra=60922000000)
       -> Returning structured response: respuesta='El revenue (ingresos totales) de NVIDIA en el ejercicio fiscal 2024 (FY2024) fue de 60.922.000.000 USD (60.922 millones de dólares).' cifra=60922000000.0 uni

### 6.2 Comparativa multi-step

Una comparativa con explicación justifica el comportamiento agentic: requiere consultar más de un ejercicio y combinar fuentes. La trayectoria permite comprobar si el modelo realmente descompuso el problema en lugar de responder por memoria paramétrica.

In [ ]:
pregunta='¿Cómo cambió el beneficio neto de Meta entre FY2024 y FY2025 y qué explica esa variación?'
r=sistema.responder(pregunta, thread_id='demo-comparativa')
miax_s2.pretty_trace(r)
print('Tools:', miax_s2.herramientas_usadas(r))

  1. list_available()
       -> AAPL (Apple Inc.): ejercicios [2024, 2025], items ['1A', '7', '7A', '8'] AMZN (AMAZON COM INC): ejercicios [2024, 2025], items ['1A', '7', '7A', '8'] GOOGL (Alphabet Inc.): ejercicios [2024, 2025], items ['1A', '7', '7A'…
  2. get_xbrl_fact(fiscal_year=2024, concept='NetIncomeLoss', ticker='META')
  3. get_xbrl_fact(concept='NetIncomeLoss', ticker='META', fiscal_year=2025)
       -> META FY2024 NetIncomeLoss = 62,360,000,000 USD (cierre 2024-12-31, 10-K)
       -> META FY2025 NetIncomeLoss = 60,458,000,000 USD (cierre 2025-12-31, 10-K)
  4. search_filings(k=5, item='7', fiscal_year=2025, query='net income decreased provision for income taxes costs and expenses 2025 compared to 2024', ticker='META')
       -> [META-2025-7-0028] META FY2025 Item 7 (similitud 0.033) Provision for income taxes  Year Ended December 31, 2025		2024		2023		2025 vs 2024 % change		2024 vs 2023 % change (in millions, except percentages) Provision for i…
  5. RespuestaFinanciera(cif

## 7. Tres evaluadores automáticos

La práctica considera fallo acertar por el camino equivocado. Por eso se evalúan tres dimensiones independientes:

1. **cita**: el `chunk_id` existe y la cita completa aparece en ese chunk;
2. **cifra**: la magnitud estructurada coincide con el ground truth XBRL dentro del 1 %;
3. **trayectoria**: se utilizaron todas las herramientas declaradas como esperadas.

Los criterios no aplicables se dejan como `None`, para que no entren en el denominador de la métrica correspondiente.

In [ ]:
assert set(sistema.EVALUADORES)=={'cita','cifra','trayectoria'}
print('Evaluadores:', list(sistema.EVALUADORES))

Evaluadores: ['cita', 'cifra', 'trayectoria']


## 8. Baseline frente a sistema final

La comparación principal usa el mismo golden set y el mismo modelo. El baseline es el agente de la sesión 1 proporcionado en `miax_s2.baseline()`: cuatro tools y búsqueda densa. El final añade retrieval híbrido, reescritura y guardrails.

Se reportan aciertos por dimensión, `recall@5`, coste medio, latencia media y llamadas a herramienta. Los CSV por pregunta se guardan en `resultados/` para que las tablas sean regenerables.

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# Baseline docente con las cuatro tools y búsqueda densa con filtros de la sesión 2.
agente_base=miax_s2.baseline(modelo=sistema.MODELO)

def responder_base(pregunta, thread_id=None):
    t0=time.perf_counter()
    r=agente_base.invoke({'messages':[{'role':'user','content':pregunta}]},
                         config={'configurable':{'thread_id':thread_id or f'base-{time.time_ns()}'}})
    dt=time.perf_counter()-t0
    tin,tout=sistema.tokens_de(r)
    return {**r,'latencia_s':dt,'tokens_entrada':tin,'tokens_salida':tout,'coste_usd':sistema.coste_estimado(r)}

def evaluar_con(preguntas, responder_fn, version):
    filas=[]
    for item in preguntas:
        fila={'id':item['id'],'familia':item.get('familia')}
        try:
            r=responder_fn(item['pregunta'], thread_id=f"{version}-{item['id']}")
            fila.update(latencia_s=r['latencia_s'],coste_usd=r['coste_usd'],
                        llamadas=len(miax_s2.herramientas_usadas(r)))
            for n,fn in sistema.EVALUADORES.items(): fila[n]=fn(item,r)
            if item.get('ancla_texto'):
                if version=='baseline':
                    rec=sistema.buscar_denso_baseline(item['pregunta'],item.get('ticker'),item.get('fiscal_year'),item.get('item_esperado'),5)
                else:
                    q=sistema.reescribir_consulta(item['pregunta'])
                    rec=sistema.buscar_hibrido(q,item.get('ticker'),item.get('fiscal_year'),item.get('item_esperado'),5)
                fila['recall@5']=miax_s2.acierta(item,rec)
            else: fila['recall@5']=None
        except Exception as e:
            fila['error']=f'{type(e).__name__}: {e}'
        filas.append(fila)
    return pd.DataFrame(filas)

RESULTADOS_DIR=PROJECT_DIR/'resultados'; RESULTADOS_DIR.mkdir(exist_ok=True)
t_base=evaluar_con(golden,responder_base,'baseline')
t_final=evaluar_con(golden,sistema.responder,'final')
t_base.to_csv(RESULTADOS_DIR/'resultados_baseline.csv',index=False)
t_final.to_csv(RESULTADOS_DIR/'resultados_final.csv',index=False)
comparacion=pd.concat([sistema.resumir_resultados(t_base,'baseline'),sistema.resumir_resultados(t_final,'final')],ignore_index=True)
display(comparacion.style.format({'cita':'{:.1%}','cifra':'{:.1%}','trayectoria':'{:.1%}','recall@5':'{:.1%}',
                                  'coste_medio_usd':'${:.5f}','latencia_media_s':'{:.2f}','llamadas_por_pregunta':'{:.2f}'}))

,version,cita,cifra,trayectoria,recall@5,coste_medio_usd,latencia_media_s,llamadas_por_pregunta
0,baseline,87.5%,100.0%,100.0%,46.2%,$0.01170,8.16,4.65
1,final,100.0%,100.0%,100.0%,84.6%,$0.01216,10.62,4.35


### 8.1 Resultados por familia y regresiones

El promedio global puede ocultar que una mejora ayude a extractivas y perjudique a numéricas. Por eso se conserva el detalle por pregunta y se muestran explícitamente tanto mejoras como regresiones. Un arreglo razonable que no mueve la métrica también es un resultado experimental válido.

In [ ]:
cols=['id','familia','cita','cifra','trayectoria','recall@5','coste_usd','latencia_s','llamadas']
detalle=t_base[cols].merge(t_final[cols],on=['id','familia'],suffixes=('_base','_final'))
display(detalle)

filas=[]
for fam in sorted(detalle.familia.dropna().unique()):
    d=detalle[detalle.familia==fam]; fila={'familia':fam,'n':len(d)}
    for met in ['cita','cifra','trayectoria','recall@5']:
        for ver in ['base','final']:
            x=d[f'{met}_{ver}'].dropna(); fila[f'{met}_{ver}']=x.astype(float).mean() if len(x) else np.nan
    filas.append(fila)
display(pd.DataFrame(filas))

for met in ['cita','cifra','trayectoria','recall@5']:
    b,f=f'{met}_base',f'{met}_final'
    mejoran=detalle[(detalle[b]==False)&(detalle[f]==True)].id.tolist()
    empeoran=detalle[(detalle[b]==True)&(detalle[f]==False)].id.tolist()
    print(f'{met}: mejoran={mejoran} · empeoran={empeoran}')

,id,familia,cita_base,cifra_base,trayectoria_base,recall@5_base,coste_usd_base,latencia_s_base,llamadas_base,cita_final,cifra_final,trayectoria_final,recall@5_final,coste_usd_final,latencia_s_final,llamadas_final
0,g3-001,numerica,None,True,True,None,0.004015,3.650900,2,None,True,True,None,0.004635,3.340558,3
1,g3-002,numerica,True,True,True,None,0.007309,6.469204,4,None,True,True,None,0.005538,4.365389,3
2,g3-003,numerica,True,True,True,None,0.009973,5.225801,4,None,True,True,None,0.004933,3.718204,3
3,g3-004,numerica,True,None,True,None,0.016041,9.371004,5,True,None,True,None,0.036327,26.157924,7
4,g3-005,numerica,None,True,True,None,0.016157,14.843995,4,True,True,True,None,0.011139,12.852135,4
5,g3-006,numerica,None,True,True,None,0.004337,3.734867,3,None,True,True,None,0.004868,8.253063,3
6,g3-007,numerica,None,True,True,None,0.005896,4.260722,5,None,True,True,None,0.006265,4.106930,5
7,g3-008,extractiva,False,None,True,False,0.012116,8.509016,3,True,None,True,False,0.016261,18.245032,3
8,g3-009,extractiva,True,None,True,True,0.011886,12.839023,4,True,None,True,True,0.007533,14.043428,3
9,g3-010,extractiva,True,None,True,False,0.013077,6.395618,4,True,None,True,True,0.013282,9.419350,4


,familia,n,cita_base,cita_final,cifra_base,cifra_final,trayectoria_base,trayectoria_final,recall@5_base,recall@5_final
0,comparativa,6,0.833333,1.0,NaN,NaN,1.0,1.0,0.500000,0.833333
1,extractiva,7,0.857143,1.0,NaN,NaN,1.0,1.0,0.428571,0.857143
2,numerica,7,1.000000,1.0,1.0,1.0,1.0,1.0,NaN,NaN


cita: mejoran=['g3-008', 'g3-015'] · empeoran=[]
cifra: mejoran=[] · empeoran=[]
trayectoria: mejoran=[] · empeoran=[]
recall@5: mejoran=['g3-010', 'g3-012', 'g3-014', 'g3-019', 'g3-020'] · empeoran=[]


## 9. Interfaces del hold-out y tests finales

Las preguntas ciegas deben poder ejecutarse sin tocar el código. Las dos interfaces públicas son:

```python
from agente_10k import responder, evaluar
responder("...")
evaluar("preguntas_ciegas.jsonl")
```

La última celda comprueba el contrato mínimo de entrega. No ejecuta una nueva evaluación y, por tanto, no añade coste.

In [ ]:
assert callable(sistema.responder) and callable(sistema.evaluar)
assert [t.name for t in sistema.HERRAMIENTAS]==['list_available','get_xbrl_fact','search_filings','read_section']
assert set(sistema.EVALUADORES)=={'cita','cifra','trayectoria'}
assert miax_s2.cuadra(100.0,100.4,0.01)
assert not miax_s2.cuadra(100.0,150.0,0.01)
assert (RESULTADOS_DIR/'resultados_baseline.csv').is_file()
assert (RESULTADOS_DIR/'resultados_final.csv').is_file()
print('✓ Contrato de tools')
print('✓ responder() / evaluar()')
print('✓ tres evaluadores')
print('✓ tolerancia XBRL documentada')
print('✓ resultados baseline/final regenerables')

✓ Contrato de tools
✓ responder() / evaluar()
✓ tres evaluadores
✓ tolerancia XBRL documentada
✓ resultados baseline/final regenerables


## 10. Lectura final de la arquitectura

El sistema final no intenta resolver todos los problemas con una única técnica. Cada componente responde a un fallo observable:

| Problema | Mecanismo |
|---|---|
| compañía/año/sección incorrectos | filtros de metadatos |
| términos literales, tickers o cifras mal rankeados | BM25 + denso con RRF |
| pregunta en español frente a corpus en inglés | query rewriting |
| cifra exacta | XBRL, no prosa |
| cifra generada incompatible con XBRL | middleware de verificación |
| comparativa multi-step | agente con varias tool calls |
| bucles/coste descontrolado | límites de tool/model calls |
| respuesta correcta por camino incorrecto | evaluador de trayectoria |

La defensa de la solución debe apoyarse en las tablas generadas, no en la complejidad del diseño: **una mejora solo se considera mejora si mueve una métrica o evita un fallo concreto, y su coste y latencia se reportan junto al beneficio**.

In [ ]:
pregunta_fallo = next(p for p in golden if p['id'] == 'g3-005')
print(pregunta_fallo['pregunta'])
print('cifra_esperada:', pregunta_fallo.get('cifra_esperada'))

r = sistema.responder(pregunta_fallo['pregunta'], thread_id='debug-g3-005')
miax_s2.pretty_trace(r)
print(r['structured_response'])

¿Cuál fue la provisión por impuestos sobre beneficios de Meta en 2025?
cifra_esperada: 25474000000
  1. list_available()
       -> AAPL (Apple Inc.): ejercicios [2024, 2025], items ['1A', '7', '7A', '8'] AMZN (AMAZON COM INC): ejercicios [2024, 2025], items ['1A', '7', '7A', '8'] GOOGL (Alphabet Inc.): ejercicios [2024, 2025], items ['1A', '7', '7A'…
  2. get_xbrl_fact(ticker='META', concept='IncomeTaxExpenseBenefit', fiscal_year=2025)
       -> META no reportó 'IncomeTaxExpenseBenefit' en FY2025. Conceptos disponibles: ['Assets', 'CashAndCashEquivalentsAtCarryingValue', 'EarningsPerShareBasic', 'EarningsPerShareDiluted', 'Liabilities', 'NetCashProvidedByUsedInO…
  3. search_filings(fiscal_year=2025, ticker='META', query='"provision for income taxes" "income before provision for income taxes" Consolidated Statements of Income', item='8')
       -> [META-2025-8-0079] META FY2025 Item 8 (similitud 0.033) Note 14. Income Taxes  The components of income before provision for income taxes ar

In [ ]:
t_base=evaluar_con(golden,responder_base,'baseline')
t_final=evaluar_con(golden,sistema.responder,'final')
t_base.to_csv(RESULTADOS_DIR/'resultados_baseline.csv',index=False)
t_final.to_csv(RESULTADOS_DIR/'resultados_final.csv',index=False)
comparacion=pd.concat([sistema.resumir_resultados(t_base,'baseline'),sistema.resumir_resultados(t_final,'final')],ignore_index=True)
display(comparacion.style.format({'cita':'{:.1%}','cifra':'{:.1%}','trayectoria':'{:.1%}','recall@5':'{:.1%}',
                                  'coste_medio_usd':'${:.5f}','latencia_media_s':'{:.2f}','llamadas_por_pregunta':'{:.2f}'}))

,version,cita,cifra,trayectoria,recall@5,coste_medio_usd,latencia_media_s,llamadas_por_pregunta
0,baseline,88.2%,100.0%,100.0%,46.2%,$0.01640,1.63,5.65
1,final,100.0%,100.0%,100.0%,84.6%,$0.01695,1.74,5.35


# Tests Andrea

In [ ]:
# Localizar chunks marcados como tabla
chunks_con_tabla = df_chunks[df_chunks["contiene_tabla"] == True] if "contiene_tabla" in df_chunks.columns else None

if chunks_con_tabla is not None:
    print(f"Total de chunks con tabla: {len(chunks_con_tabla)}")
    print(chunks_con_tabla.groupby(["ticker", "item"]).size())
else:
    print("La columna 'contiene_tabla' no está en df_chunks — comprobar en df_meta")
    print(df_meta.columns.tolist())

Total de chunks con tabla: 721
ticker  item
AAPL    7        10
        7A        2
        8        65
AMZN    1A        2
        7        16
        7A        6
        8        77
GOOGL   1A       32
        7        38
        7A        8
        8       133
META    7        26
        8        81
MSFT    7        14
        7A        2
        8       100
NVDA    7        16
        8        93
dtype: int64


In [ ]:
# Coge un chunk de tabla concreto para inspeccionar su contenido
ejemplo_tabla = chunks_con_tabla.iloc[0]
print(f"chunk_id: {ejemplo_tabla['chunk_id']} | {ejemplo_tabla['ticker']} FY{ejemplo_tabla['fiscal_year']} Item {ejemplo_tabla['item']}")
print(ejemplo_tabla["texto"])

chunk_id: NVDA-2024-7-0006 | NVDA FY2024 Item 7
Year Ended
Jan 28, 2024		Jan 29, 2023		Change
($ in millions, except per share data)
Revenue	$	60,922			$	26,974			Up 126%
Gross margin	72.7	%		56.9	%		Up 15.8 pts
Operating expenses	$	11,329			$	11,132			Up 2%
Operating income	$	32,972			$	4,224			Up 681%
Net income	$	29,760			$	4,368			Up 581%
Net income per diluted share	$	11.93			$	1.74			Up 586%

We specialize in markets where our computing platforms can provide tremendous acceleration for applications. These platforms incorporate processors, interconnects, software, algorithms, systems, and services to deliver unique value. Our platforms address four large markets where our expertise is critical: Data Center, Gaming, Professional Visualization, and Automotive.

Revenue for fiscal year 2024 was $60.9 billion, up 126% from a year ago.

Data Center revenue for fiscal year 2024 was up 217%. Strong demand was driven by enterprise software and consumer internet applications, and multiple 

# JSON

In [ ]:
import json

prueba_ciega = [
    {"id": "test-001", "familia": "numerica", "ticker": "AAPL", "fiscal_year": 2024,
     "pregunta": "¿Cuál fue el beneficio neto de Apple en FY2024?",
     "cifra_esperada": None, "unidad": "USD", "concept_xbrl": "NetIncomeLoss",
     "item_esperado": None, "ancla_texto": None, "chunk_id_esperado": None,
     "herramienta_esperada": ["get_xbrl_fact"]},

    {"id": "test-002", "familia": "extractiva", "ticker": "NVDA", "fiscal_year": 2025,
     "pregunta": "¿Qué dice NVIDIA sobre la competencia en el mercado de GPUs?",
     "cifra_esperada": None, "unidad": None, "concept_xbrl": None,
     "item_esperado": "1A", "ancla_texto": None, "chunk_id_esperado": None,
     "herramienta_esperada": ["search_filings"]},

    {"id": "test-003", "familia": "numerica", "ticker": "NVDA", "fiscal_year": 2024,
     "pregunta": "Según la tabla comparativa de NVIDIA, ¿cuál fue el margen bruto (gross margin) en el ejercicio fiscal 2024 frente al 2023?",
     "cifra_esperada": 72.7, "unidad": "porcentaje", "concept_xbrl": None,
     "item_esperado": "7", "ancla_texto": "Gross margin\t72.7\t%\t\t56.9\t%\t\tUp 15.8 pts",
     "chunk_id_esperado": "NVDA-2024-7-0006", "herramienta_esperada": ["search_filings"]},

    {"id": "test-004", "familia": "numerica", "ticker": "NVDA", "fiscal_year": 2024,
     "pregunta": "¿Cuánto crecieron los ingresos por diluted share de NVIDIA (net income per diluted share) entre el año fiscal 2023 y 2024?",
     "cifra_esperada": 585.63, "unidad": "porcentaje", "concept_xbrl": None,
     "item_esperado": "7", "ancla_texto": "Net income per diluted share\t$\t11.93\t\t\t$\t1.74\t\t\tUp 586%",
     "chunk_id_esperado": "NVDA-2024-7-0006", "herramienta_esperada": ["search_filings"]},

    {"id": "test-005", "familia": "numerica", "ticker": "AMZN", "fiscal_year": 2025,
     "pregunta": "¿Cuál fue el resultado operativo combinado de AWS y Norteamérica de Amazon en FY2025?",
     "cifra_esperada": 75225000000, "unidad": "USD", "concept_xbrl": None,
     "item_esperado": "7", "ancla_texto": "Operating income by segment is as follows (in millions):",
     "chunk_id_esperado": "AMZN-2025-7-0019", "herramienta_esperada": ["search_filings"]},

    {"id": "test-006", "familia": "comparativa", "ticker": "META", "fiscal_year": 2025,
     "pregunta": "Entre Meta, Apple y Amazon, ¿qué empresa tuvo el mayor gasto en I+D en FY2025?",
     "cifra_esperada": None, "unidad": "USD", "concept_xbrl": None,
     "item_esperado": None, "ancla_texto": None, "chunk_id_esperado": None,
     "herramienta_esperada": ["get_xbrl_fact"]},

    {"id": "test-007", "familia": "numerica", "ticker": "TSLA", "fiscal_year": 2025,
     "pregunta": "¿Cuál fue el beneficio neto de Tesla en FY2025?",
     "cifra_esperada": None, "unidad": "USD", "concept_xbrl": None,
     "item_esperado": None, "ancla_texto": None, "chunk_id_esperado": None,
     "herramienta_esperada": ["list_available"]},

    {"id": "test-008", "familia": "extractiva", "ticker": "GOOGL", "fiscal_year": 2025,
     "pregunta": "¿Va bien Alphabet?",
     "cifra_esperada": None, "unidad": None, "concept_xbrl": None,
     "item_esperado": None, "ancla_texto": None, "chunk_id_esperado": None,
     "herramienta_esperada": []},
]

with open("prueba_ciega.jsonl", "w", encoding="utf-8") as f:
    for p in prueba_ciega:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

print(f"prueba_ciega.jsonl creado con {len(prueba_ciega)} preguntas")

prueba_ciega.jsonl creado con 8 preguntas


In [ ]:
resultado_prueba = sistema.evaluar("prueba_ciega.jsonl", salida_csv="prueba_ciega_resultados.csv")
display(resultado_prueba)

,id,familia,latencia_s,coste_usd,tokens_entrada,tokens_salida,llamadas,cita,cifra,trayectoria,recall@5,respuesta
0,test-001,numerica,1.394782,0.005065,4314,488,3,None,None,True,None,El beneficio neto de Apple en el ejercicio fis...
1,test-002,extractiva,2.064651,0.066363,73729,2951,9,True,None,True,None,En su informe 10-K para el ejercicio fiscal 20...
2,test-003,numerica,1.309712,0.021920,20851,1675,6,True,True,True,True,"Según la tabla comparativa de NVIDIA (Item 7),..."
3,test-004,numerica,1.899479,0.066816,73008,3216,11,False,False,True,True,El beneficio neto por acción diluida (net inco...
4,test-005,numerica,1.348953,0.029021,27050,2329,7,False,True,True,True,"En el ejercicio fiscal 2025, el resultado oper..."
5,test-006,comparativa,1.938587,0.036763,37548,2294,9,True,None,True,None,De las empresas que reportan una partida espec...
6,test-007,numerica,0.925274,0.005152,4585,457,3,None,None,True,None,Tesla (TSLA) no se encuentra disponible en el ...
7,test-008,extractiva,1.378903,0.023485,23279,1607,10,True,None,True,None,"Sí, Alphabet muestra un sólido desempeño finan..."


In [ ]:
import shutil
shutil.copy("prueba_ciega.jsonl", "/content/drive/MyDrive/MIAX_Taller_NLP/prueba_ciega.jsonl")
shutil.copy("prueba_ciega_resultados.csv", "/content/drive/MyDrive/MIAX_Taller_NLP/prueba_ciega_resultados.csv")
print("Copiado a Drive.")

Copiado a Drive.


In [ ]:
r4 = sistema.responder(
    "¿Cuánto crecieron los ingresos por diluted share de NVIDIA (net income per diluted share) entre el año fiscal 2023 y 2024?",
    thread_id="test-004-verificar2"
)
sr4 = r4["structured_response"]
print("cifra devuelta:", sr4.cifra)
print("cifra_esperada:", 585.63)
print("chunk_id devuelto:", sr4.chunk_id)
print("chunk_id_esperado:", "NVDA-2024-7-0006")
print("cita devuelta:", repr(sr4.cita))

cifra devuelta: 586.0
cifra_esperada: 585.63
chunk_id devuelto: NVDA-2024-7-0006
chunk_id_esperado: NVDA-2024-7-0006
cita devuelta: 'Net income per diluted share\t$\t11.93\t\t\t$\t1.74\t\t\tUp 586%'


In [ ]:
r5b = sistema.responder(
    "¿Cuál fue el resultado operativo combinado de AWS y Norteamérica de Amazon en FY2025?",
    thread_id="test-005-verificar2"
)
sr5b = r5b["structured_response"]
print("chunk_id devuelto:", sr5b.chunk_id)
print("chunk_id_esperado:", "AMZN-2025-7-0019")
print("cita devuelta:", repr(sr5b.cita))

chunk_id devuelto: AMZN-2025-7-0019
chunk_id_esperado: AMZN-2025-7-0019
cita devuelta: 'Operating income by segment is as follows (in millions): ... North America $ 29,619 ... AWS 45,606'


In [ ]:
print(miax_s2.cuadra(586.0, 585.63, 0.01))

True


In [ ]:
with open("prueba_ciega.jsonl", encoding="utf-8") as f:
    preguntas_cargadas = [json.loads(l) for l in f if l.strip()]

for p in preguntas_cargadas:
    if p["id"] == "test-004":
        print(p["cifra_esperada"])